# Curriculum-Based Long/Short CoT Distillation for Code Generation

**Thesis Notebook — RunPod-ready, Qwen2.5-1.5B-Instruct, no Unsloth**

## Research question
Does *curriculum-based mixing* of long and short chain-of-thought training data improve small-model code generation over (a) static mixing and (b) pure long/short CoT, replicating and extending the Small Model Learnability Gap of Li et al. (2025, arXiv:2502.12143) from math reasoning to code?

## Experimental design

| # | Condition         | Training data                                          | Schedule             |
|---|-------------------|--------------------------------------------------------|----------------------|
| 1 | Base              | none (zero-shot reference)                             | —                    |
| 2 | Pure Short CoT    | 100% short                                             | 3 epochs             |
| 3 | Pure Long CoT     | 100% long                                              | 3 epochs             |
| 4 | Static Mix α=0.2  | 20% long / 80% short (per-problem random assignment)   | 3 epochs             |
| 5 | Static Mix α=0.5  | 50% long / 50% short                                   | 3 epochs             |
| 6 | Curriculum        | Stage 1: 0% long → Stage 2: 50% → Stage 3: 80%         | 3 stages × 1 epoch   |

All trained-model conditions use **identical hyperparameters** — only the data composition varies.

## Evaluation

- **HumanEval** (164 problems) — both *training-format* (chat-templated) and *direct-completion* prompts
- **MBPP held-out** (~195 problems, 20% deterministic split, never seen during training) — in-domain generalization
- **HumanEval+** (optional flag) — robustness

Metric: pass@1 with greedy decoding, `max_new_tokens=3072`, execution-based grading with timeout.

## Methodology compliance with diagnosed issues

This notebook addresses every issue from the diagnosis:

1. ✅ All baselines included (Pure Short, Pure Long, two static mixes, Curriculum, Base).
2. ✅ Curriculum endpoint kept at 0 → 50 → 80% per thesis design — Static α=0.2 condition independently tests the paper's optimum so we can compare.
3. ✅ Both chat-template and direct-completion HumanEval prompts evaluated.
4. ✅ Robust code extractor with four fallback strategies.
5. ✅ `max_new_tokens=3072`.
6. ✅ Deterministic 80/20 MBPP split (seed=42), held-out never seen during training.
7. ✅ Multi-benchmark evaluation (HumanEval + MBPP held-out, HumanEval+ optional).
8. ✅ Switched to `Qwen/Qwen2.5-1.5B-Instruct` (general, not Coder variant) for headroom.
9. ✅ Pure-Long vs Pure-Short comparison directly tests the gap.
10. ✅ Single seed by default; multi-seed loop is in place — run repeats on key conditions if time permits.

## GPU-hour budget (estimates for L40S 48 GB / RTX A5000 24 GB without Unsloth)

| Phase                                                       | Wall-clock (L40S)   |
|-------------------------------------------------------------|---------------------|
| Pure Short CoT — 3 epochs                                   | ~25 min             |
| Pure Long CoT — 3 epochs                                    | ~75 min             |
| Static Mix α=0.2 — 3 epochs                                 | ~35 min             |
| Static Mix α=0.5 — 3 epochs                                 | ~50 min             |
| Curriculum — 3 stages × 1 epoch                             | ~50 min             |
| **Training subtotal**                                       | **~4 hr**           |
| Evaluation — 6 models × HE × {chat, direct} + MBPP-heldout  | **~12–15 hr**       |
| **End-to-end**                                              | **~16–19 hr**       |

A5000 (24 GB) will be ~30–50% slower; the workflow runs without quantization in bf16 either way.

## Resume support

- **Training**: skips a condition if its adapter directory already contains `adapter_model.safetensors`.
- **Evaluation**: saves raw generations to JSONL — re-grading does not require regeneration.


---
## 1. Environment setup

This notebook is designed for the **Trelis fine-tuning RunPod template** (PyTorch 2.2.0 + CUDA 12.1 + Python 3.10 + Ubuntu 22.04, with SSH terminal exposed). Run the install cell once on a fresh pod; subsequent kernel restarts skip the reinstall via the `_pkgs_installed_v1` flag file in `/workspace`.

Library versions are pinned for compatibility with PyTorch 2.2 / CUDA 12.1.


In [ ]:
# === one-shot dependency install (idempotent) ===
import os, subprocess, sys, pathlib

PKG_FLAG = pathlib.Path("/workspace/.pkgs_installed_v1")

REQUIREMENTS = [
    "transformers==4.46.3",
    "peft==0.13.2",
    "trl==0.12.2",
    "accelerate==0.34.2",
    "datasets==3.1.0",
    "bitsandbytes==0.44.1",
    "sentencepiece==0.2.0",
    "protobuf==4.25.5",
    "evaluate==0.4.3",
    "tqdm",
    "pandas",
    "numpy",
    "matplotlib",
]

if not PKG_FLAG.exists():
    print("Installing pinned dependencies (one-time, ~3-5 min)…")
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + REQUIREMENTS
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout[-2000:] if res.stdout else "")
    if res.returncode != 0:
        print("STDERR:", res.stderr[-2000:])
        raise RuntimeError("pip install failed — see stderr above")
    PKG_FLAG.parent.mkdir(parents=True, exist_ok=True)
    PKG_FLAG.touch()
    print("✓ Install complete. **You may need to restart the kernel once** for the new package versions to load.")
else:
    print(f"✓ Already installed (flag at {PKG_FLAG}) — skipping pip install.")


In [ ]:
# === environment sanity check ===
import torch, transformers, peft, trl, accelerate, datasets, bitsandbytes
import platform

print(f"Python         : {platform.python_version()}")
print(f"torch          : {torch.__version__}  (CUDA build: {torch.version.cuda})")
print(f"transformers   : {transformers.__version__}")
print(f"peft           : {peft.__version__}")
print(f"trl            : {trl.__version__}")
print(f"accelerate     : {accelerate.__version__}")
print(f"datasets       : {datasets.__version__}")
print(f"bitsandbytes   : {bitsandbytes.__version__}")
print()
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU            : {p.name}")
    print(f"VRAM           : {p.total_memory / 1024**3:.1f} GiB")
    print(f"Capability     : sm_{p.major}{p.minor}")
else:
    raise RuntimeError("No GPU detected — this notebook requires a CUDA GPU.")


---
## 2. Master configuration

Every knob lives in a single `CONFIG` dict so each downstream cell reads from one source of truth. This is also useful as a thesis appendix table.


In [ ]:
import os
from pathlib import Path

# Set HF cache to /workspace so it survives pod restarts when using a network volume
os.environ.setdefault("HF_HOME", "/workspace/.hf_cache")
os.environ.setdefault("TRANSFORMERS_CACHE", "/workspace/.hf_cache/transformers")
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

CONFIG = {
    # --------- model ---------
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",   # general Instruct, NOT Coder — gives headroom

    # --------- paths ---------
    "data_root":    Path("/workspace/data"),
    "master_jsonl": Path("/workspace/data/mbpp_curriculum_master.jsonl"),
    "splits_dir":   Path("/workspace/data/splits"),
    "datasets_dir": Path("/workspace/data/training_sets"),
    "adapters_dir": Path("/workspace/runs/adapters"),
    "outputs_dir":  Path("/workspace/runs/eval_outputs"),
    "results_dir":  Path("/workspace/runs/results"),

    # --------- data split ---------
    "split_seed":   42,
    "test_frac":    0.20,

    # --------- LoRA ---------
    "lora_r":       16,
    "lora_alpha":   32,
    "lora_dropout": 0.05,
    "lora_target":  "all-linear",     # all linear modules except embed/lm_head

    # --------- training (SHARED across all conditions — only data varies) ---------
    "learning_rate":          2.0e-4,
    "lr_scheduler":           "cosine",
    "warmup_ratio":           0.05,
    "per_device_batch":       4,
    "grad_accum":             4,        # effective batch = 16
    "max_seq_len":            3072,     # accommodates long CoT + code
    "weight_decay":           0.0,
    "optim":                  "adamw_torch",
    "bf16":                   True,
    "logging_steps":          5,
    "save_strategy":          "no",     # we save final adapter manually
    "gradient_checkpointing": True,

    # --------- conditions ---------
    "conditions": [
        # name              type         alpha    epochs   stages
        ("pure_short",     "static",     0.0,     3,       None),
        ("pure_long",      "static",     1.0,     3,       None),
        ("static_mix_a02", "static",     0.2,     3,       None),
        ("static_mix_a05", "static",     0.5,     3,       None),
        ("curriculum",     "curriculum", None,    None,    [(0.0, 1), (0.5, 1), (0.8, 1)]),
    ],

    # --------- evaluation ---------
    "max_new_tokens":         3072,
    "do_sample":              False,    # greedy
    "exec_timeout":           10,       # seconds per code execution
    "eval_humaneval":         True,
    "eval_humaneval_plus":    False,    # flip True if installed
    "eval_mbpp_heldout":      True,
    "eval_prompt_formats":    ["chat", "direct"],   # both for HumanEval; MBPP uses chat only

    # --------- seeds ---------
    "training_seeds":         [42],     # extend to [42, 1337] etc. for variance estimates
    "eval_seed":              42,

    # --------- smoke test ---------
    "smoke_n_train":          8,
    "smoke_n_eval":           4,
    "smoke_max_steps":        4,
}

# materialize directories
for key in ("data_root", "splits_dir", "datasets_dir",
            "adapters_dir", "outputs_dir", "results_dir"):
    CONFIG[key].mkdir(parents=True, exist_ok=True)

print("CONFIG loaded.")
print(f"  Adapters    → {CONFIG['adapters_dir']}")
print(f"  Eval outs   → {CONFIG['outputs_dir']}")
print(f"  Results     → {CONFIG['results_dir']}")


In [ ]:
# === imports + reproducibility ===
import json, random, re, gc, time, hashlib, traceback, subprocess, tempfile, sys
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    set_seed, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def reseed(s: int) -> None:
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    set_seed(s)

reseed(CONFIG["split_seed"])
print("Imports OK; seeded with", CONFIG["split_seed"])


---
## 3. Data preparation

### 3.1 Load master file and build the deterministic 80/20 split

The master JSONL `mbpp_curriculum_master.jsonl` has 974 records, each with:

- `task_id`   — MBPP problem id
- `prompt`    — natural-language problem (used as user turn)
- `short_cot` — original MBPP solution (assistant turn for "short" condition)
- `long_cot`  — DeepSeek-Coder generated reasoning + code (assistant turn for "long")
- `test_list` — assertion list for execution-based grading on MBPP held-out

The 80/20 split is **deterministic** (seeded shuffle) and persisted to disk so every condition sees the **identical** train set and the held-out test set is **never** seen during training.

> **Note**: Any prior `epoch_1.jsonl / epoch_2.jsonl / epoch_3.jsonl` files from the original setup are intentionally ignored — they were built before the train/test split was introduced. All training datasets are rebuilt below from the master file using only the train split.


In [ ]:
# === sanity-check the master file ===
master_path = CONFIG["master_jsonl"]
assert master_path.exists(), (
    f"Master file not found at {master_path}. "
    "Place mbpp_curriculum_master.jsonl there before running this cell."
)

records = [json.loads(l) for l in master_path.read_text().splitlines() if l.strip()]
print(f"Loaded {len(records)} records from {master_path.name}")

# verify required fields
required = ("task_id", "prompt", "short_cot", "long_cot")
missing = [r["task_id"] for r in records if not all(k in r for k in required)]
if missing:
    print(f"⚠ {len(missing)} records missing required keys; first 5 ids: {missing[:5]}")
else:
    print(f"✓ All records have required keys: {required}")

# test_list may live under different keys depending on how master file was generated
HAS_TESTS = all("test_list" in r and r["test_list"] for r in records)
if HAS_TESTS:
    print(f"✓ All records carry test_list — MBPP held-out execution grading enabled.")
else:
    n_with = sum(1 for r in records if r.get("test_list"))
    print(f"⚠ Only {n_with}/{len(records)} records have test_list. "
          "MBPP held-out grading will skip records without tests.")


In [ ]:
# === deterministic 80/20 split, persisted ===
split_seed = CONFIG["split_seed"]
test_frac  = CONFIG["test_frac"]

train_path = CONFIG["splits_dir"] / "train.jsonl"
test_path  = CONFIG["splits_dir"] / "test.jsonl"

if train_path.exists() and test_path.exists():
    train_records = [json.loads(l) for l in train_path.read_text().splitlines()]
    test_records  = [json.loads(l) for l in test_path.read_text().splitlines()]
    print(f"✓ Loaded existing split  →  train={len(train_records)}  test={len(test_records)}")
else:
    rng = random.Random(split_seed)
    shuffled = records[:]
    rng.shuffle(shuffled)
    n_test = int(round(len(shuffled) * test_frac))
    test_records  = sorted(shuffled[:n_test], key=lambda r: r["task_id"])
    train_records = sorted(shuffled[n_test:], key=lambda r: r["task_id"])

    with train_path.open("w") as f:
        for r in train_records: f.write(json.dumps(r) + "\n")
    with test_path.open("w") as f:
        for r in test_records:  f.write(json.dumps(r) + "\n")
    print(f"✓ Created split   →  train={len(train_records)}  test={len(test_records)}")
    print(f"  saved to {CONFIG['splits_dir']}")

# integrity hash (so any accidental change is loud)
def _hash(records):
    return hashlib.sha256(
        "".join(str(r["task_id"]) for r in records).encode()
    ).hexdigest()[:12]

print(f"  train hash: {_hash(train_records)}")
print(f"  test  hash: {_hash(test_records)}")


### 3.2 Build per-condition training datasets

We use the **Qwen2.5 chat template** to format every example. The user turn is the MBPP prompt; the assistant turn is either the short or long CoT, depending on the condition.

**Static Mix construction** (matches Li et al.'s Mix-Long): for each problem, with probability α, that problem is delivered in long-CoT form; otherwise in short-CoT form. Each problem appears exactly once per epoch (no duplication), and the random assignment is seeded so it is identical across runs of the same condition.

**Curriculum construction**: three separate datasets are built — one per stage — using the same α-based per-problem random assignment but with α = 0.0 / 0.5 / 0.8.


In [ ]:
# === load tokenizer once (also used during training and eval) ===
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"Tokenizer loaded — vocab={len(tokenizer)}, eos='{tokenizer.eos_token}'")


In [ ]:
# === example formatter and dataset builders ===
def format_example(record: Dict[str, Any], use_long: bool) -> str:
    """Apply Qwen2.5 chat template to one (problem, solution) pair."""
    user = record["prompt"].strip()
    assistant = (record["long_cot"] if use_long else record["short_cot"]).strip()

    messages = [
        {"role": "user",      "content": user},
        {"role": "assistant", "content": assistant},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


def build_static_dataset(train_records, alpha: float, seed: int) -> Dataset:
    """One example per problem; with prob alpha use long_cot else short_cot."""
    rng = random.Random(seed)
    rows = []
    for rec in train_records:
        use_long = rng.random() < alpha
        rows.append({
            "task_id": rec["task_id"],
            "is_long": int(use_long),
            "text":    format_example(rec, use_long),
        })
    return Dataset.from_list(rows)


def build_curriculum_stages(train_records, stage_alphas, seed: int) -> List[Dataset]:
    """One Dataset per curriculum stage; each stage independently sampled."""
    return [
        build_static_dataset(train_records, alpha=a, seed=seed + 1000 * idx)
        for idx, a in enumerate(stage_alphas)
    ]


In [ ]:
# === build, cache, and report all training datasets ===
ds_dir = CONFIG["datasets_dir"]
ds_dir.mkdir(parents=True, exist_ok=True)

training_sets = {}    # condition_name -> list of Dataset (each item is one stage)

# Static-mix conditions
for name, kind, alpha, epochs, stages in CONFIG["conditions"]:
    if kind != "static":
        continue
    path = ds_dir / f"{name}.jsonl"
    if path.exists():
        ds = Dataset.from_json(str(path))
    else:
        ds = build_static_dataset(train_records, alpha=alpha, seed=CONFIG["split_seed"])
        ds.to_json(str(path))
    training_sets[name] = [ds]
    n_long = sum(ds["is_long"])
    print(f"[static]    {name:18s}  alpha={alpha:.2f}  n={len(ds)}  long={n_long}  short={len(ds)-n_long}")

# Curriculum stages
for name, kind, _, _, stages in CONFIG["conditions"]:
    if kind != "curriculum":
        continue
    stage_alphas = [a for (a, _) in stages]
    stage_paths = [ds_dir / f"{name}_stage{i+1}.jsonl" for i in range(len(stages))]
    if all(p.exists() for p in stage_paths):
        stage_ds = [Dataset.from_json(str(p)) for p in stage_paths]
    else:
        stage_ds = build_curriculum_stages(train_records, stage_alphas, seed=CONFIG["split_seed"])
        for d, p in zip(stage_ds, stage_paths):
            d.to_json(str(p))
    training_sets[name] = stage_ds
    for i, (d, a) in enumerate(zip(stage_ds, stage_alphas), 1):
        n_long = sum(d["is_long"])
        print(f"[curric.]   {name}_stage{i:d}  alpha={a:.2f}  n={len(d)}  long={n_long}  short={len(d)-n_long}")

print()
print(f"All training datasets ready -> {ds_dir}")


---
## 4. Training infrastructure

### 4.1 Design notes

- **LoRA** with `r=16`, `α=32`, `dropout=0.05`, target = all linear modules. This is sufficient adaptation capacity for a 1.5B model while keeping the trainable parameter count small (~10M).
- **bf16** end-to-end on L40S (no quantization needed at 48 GB). On 24 GB GPUs, set `CONFIG["load_in_4bit"] = True` (not the default).
- **Gradient checkpointing** on to keep activation memory bounded for `max_seq_len=3072`.
- **Effective batch = 16** (per-device 4 × grad-accum 4) — same on both target GPU classes.
- **Cosine LR schedule** with 5% warmup, peak lr `2e-4`. Conservative; this is the typical operating point for LoRA on 1B-3B models.
- **Identical hyperparameters across all conditions** — only the data composition changes. This is the key methodological invariant that lets us causally attribute differences to data, not training dynamics.

### 4.2 Curriculum implementation

The curriculum runs as **three sequential `SFTTrainer.train()` calls**, each with `num_train_epochs=1`. Stage 2 loads the LoRA adapter saved by Stage 1 and trains further on that stage's data; Stage 3 likewise loads from Stage 2. This is the most faithful implementation of "curriculum across epochs" — the alternative of concatenating stages into a single dataset would let the data shuffler mix stages, which is **not** a curriculum.

### 4.3 Resume support

`train_condition()` checks for an existing `adapter_model.safetensors` in the output directory. If found, the condition is skipped. Delete the directory to force re-training.


In [ ]:
# === model loaders ===
def load_base_model(load_in_4bit: bool = False):
    """Fresh base model, ready to attach a LoRA adapter."""
    kwargs = dict(
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    if load_in_4bit:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
    model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], **kwargs)
    model.config.use_cache = False     # required for grad checkpointing
    return model


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


In [ ]:
# === core training routine: handles single-stage or sequential curriculum stages ===

def _build_lora_config():
    return LoraConfig(
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=CONFIG["lora_dropout"],
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=CONFIG["lora_target"],
    )


def _build_sft_config(out_dir: Path, num_epochs: int, seed: int, max_steps: int = -1):
    """Build SFTConfig with shared hyperparameters."""
    return SFTConfig(
        output_dir=str(out_dir / "_trainer"),
        num_train_epochs=num_epochs,
        max_steps=max_steps,
        per_device_train_batch_size=CONFIG["per_device_batch"],
        gradient_accumulation_steps=CONFIG["grad_accum"],
        learning_rate=CONFIG["learning_rate"],
        lr_scheduler_type=CONFIG["lr_scheduler"],
        warmup_ratio=CONFIG["warmup_ratio"],
        weight_decay=CONFIG["weight_decay"],
        optim=CONFIG["optim"],
        bf16=CONFIG["bf16"],
        gradient_checkpointing=CONFIG["gradient_checkpointing"],
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=CONFIG["logging_steps"],
        save_strategy=CONFIG["save_strategy"],
        report_to="none",
        seed=seed,
        data_seed=seed,
        dataset_text_field="text",
        max_seq_length=CONFIG["max_seq_len"],
        packing=False,                    # keep examples aligned to per-problem semantics
        remove_unused_columns=False,
    )


def _train_one_stage(model, dataset: Dataset, out_dir: Path,
                     num_epochs: int, seed: int, smoke_max_steps: int = -1):
    """Run one SFT stage; saves the LoRA adapter to out_dir."""
    sft_config = _build_sft_config(out_dir, num_epochs, seed, max_steps=smoke_max_steps)

    # TRL 0.12 prefers processing_class; older builds use tokenizer
    try:
        trainer = SFTTrainer(
            model=model,
            train_dataset=dataset,
            args=sft_config,
            processing_class=tokenizer,
        )
    except TypeError:
        trainer = SFTTrainer(
            model=model,
            train_dataset=dataset,
            args=sft_config,
            tokenizer=tokenizer,
        )

    train_result = trainer.train()

    # save adapter (and tokenizer for convenience)
    out_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    metrics = {
        "loss": train_result.training_loss,
        "runtime_s": train_result.metrics.get("train_runtime", None),
        "samples_per_s": train_result.metrics.get("train_samples_per_second", None),
        "n_examples": len(dataset),
        "n_epochs": num_epochs,
    }
    return trainer.model, metrics


def train_condition(name: str, kind: str, alpha,
                    epochs, stages,
                    seed: int = 42,
                    smoke: bool = False,
                    load_in_4bit: bool = False) -> Dict[str, Any]:
    """Train a single condition end-to-end. Resumes if adapter already exists."""
    cond_seed_dir = CONFIG["adapters_dir"] / f"{name}__seed{seed}"
    final_marker = cond_seed_dir / "adapter_model.safetensors"

    if final_marker.exists() and not smoke:
        print(f"[skip]  {name} (seed={seed}) — adapter already present at {cond_seed_dir}")
        return {"name": name, "seed": seed, "status": "skipped", "out_dir": str(cond_seed_dir)}

    print(f"\n{'=' * 70}\nTraining: {name}  (seed={seed}, smoke={smoke})\n{'=' * 70}")
    reseed(seed)
    free_gpu()

    # load fresh base + attach LoRA
    base = load_base_model(load_in_4bit=load_in_4bit)
    if load_in_4bit:
        from peft import prepare_model_for_kbit_training
        base = prepare_model_for_kbit_training(base)
    model = get_peft_model(base, _build_lora_config())

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {n_trainable:,} / {n_total:,} "
          f"({100*n_trainable/n_total:.2f}%)")

    cond_seed_dir.mkdir(parents=True, exist_ok=True)
    smoke_steps = CONFIG["smoke_max_steps"] if smoke else -1

    all_metrics = []
    if kind == "static":
        ds = training_sets[name][0]
        if smoke:
            ds = ds.select(range(min(CONFIG["smoke_n_train"], len(ds))))
        model, m = _train_one_stage(model, ds, cond_seed_dir, num_epochs=epochs,
                                    seed=seed, smoke_max_steps=smoke_steps)
        m["stage"] = 0
        all_metrics.append(m)

    elif kind == "curriculum":
        # Sequential stages: each loads from the previous (model object is reused).
        for stage_idx, ((stage_alpha, stage_epochs), ds) in enumerate(zip(stages, training_sets[name])):
            stage_name = f"stage{stage_idx + 1}"
            stage_dir = cond_seed_dir / stage_name
            print(f"\n  -- Curriculum {stage_name}: alpha={stage_alpha}, epochs={stage_epochs}")
            if smoke:
                ds = ds.select(range(min(CONFIG["smoke_n_train"], len(ds))))
            model, m = _train_one_stage(model, ds, stage_dir, num_epochs=stage_epochs,
                                        seed=seed, smoke_max_steps=smoke_steps)
            m["stage"] = stage_idx + 1
            m["alpha"] = stage_alpha
            all_metrics.append(m)
        # final adapter = post-stage-3 adapter; mirror it into cond_seed_dir
        final_stage_dir = cond_seed_dir / f"stage{len(stages)}"
        for f in final_stage_dir.iterdir():
            if f.is_file():
                (cond_seed_dir / f.name).write_bytes(f.read_bytes())
    else:
        raise ValueError(f"Unknown kind: {kind}")

    # cleanup
    del model, base
    free_gpu()

    summary = {
        "name": name, "seed": seed, "kind": kind,
        "out_dir": str(cond_seed_dir),
        "stage_metrics": all_metrics,
        "status": "trained",
    }
    (cond_seed_dir / "training_summary.json").write_text(json.dumps(summary, indent=2))
    return summary


---
## 5. Evaluation infrastructure

We need three pieces:

1. **Benchmark loaders** — HumanEval (164 problems) and MBPP held-out (~195 problems from our 20% test split).
2. **Robust code extractor** — parses generated text into a runnable code string with four fallback strategies for truncated outputs and mixed-content responses.
3. **Execution-based grader** — runs extracted code against test cases in a sandboxed subprocess with timeout.

These are defined first so the smoke test can use them.

### 5.1 Benchmark loaders

HumanEval is fetched from the official OpenAI repo (it's a single 164-record JSONL and is small enough to download every time). MBPP held-out is **already** persisted as `splits/test.jsonl` from Section 3.


In [ ]:
# === HumanEval loader (downloads once, caches in /workspace) ===
import urllib.request, gzip

HUMANEVAL_URL = "https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz"
HUMANEVAL_PATH = CONFIG["data_root"] / "HumanEval.jsonl"


def load_humaneval():
    if HUMANEVAL_PATH.exists():
        return [json.loads(l) for l in HUMANEVAL_PATH.read_text().splitlines() if l.strip()]
    print(f"Downloading HumanEval from {HUMANEVAL_URL} ...")
    gz_path = CONFIG["data_root"] / "HumanEval.jsonl.gz"
    urllib.request.urlretrieve(HUMANEVAL_URL, gz_path)
    with gzip.open(gz_path, "rt") as f:
        text = f.read()
    HUMANEVAL_PATH.write_text(text)
    gz_path.unlink()
    records = [json.loads(l) for l in text.splitlines() if l.strip()]
    print(f"Saved {len(records)} HumanEval problems to {HUMANEVAL_PATH}")
    return records


humaneval_problems = load_humaneval()
print(f"HumanEval: {len(humaneval_problems)} problems loaded")
print("First problem keys:", list(humaneval_problems[0].keys()))
print("Sample task_id:", humaneval_problems[0]["task_id"])


In [ ]:
# === MBPP held-out loader (uses the 20% split persisted in Section 3) ===
mbpp_heldout = test_records  # already loaded in Section 3
mbpp_with_tests = [r for r in mbpp_heldout if r.get("test_list")]
print(f"MBPP held-out: {len(mbpp_heldout)} problems "
      f"({len(mbpp_with_tests)} grader-eligible with test_list)")
if mbpp_with_tests:
    print("Sample task_id:", mbpp_with_tests[0]["task_id"])
    print("Sample tests:", mbpp_with_tests[0]["test_list"][:2])


### 5.2 Robust code extractor

The original setup used a single regex that broke on long-CoT outputs with multiple code blocks or truncated responses. The extractor below tries four strategies in order:

1. **Last complete fenced block** — for outputs with multiple ` ```python ... ``` ` blocks (common in long CoT), take the last one as the final answer.
2. **Unclosed fence** — handles outputs truncated mid-code-block (returns whatever follows the last opening fence, up to truncation).
3. **`def`/`class`/`import` heuristic** — for outputs with no code fence at all, extract from the first Python keyword to the end.
4. **Pass-through** — last resort; returns the full output. Will usually fail to execute, which is the intended outcome for malformed responses.


In [ ]:
# === robust code extractor with 4 fallback strategies ===
_FENCE_OPEN_RE = re.compile(r"```(?:python|py)?\s*\n", re.IGNORECASE)
_FENCE_BLOCK_RE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.DOTALL | re.IGNORECASE)
_PY_START_RE = re.compile(r"^\s*(def |class |import |from |async def )", re.MULTILINE)


def extract_code(text: str) -> str:
    """Best-effort extraction of Python source from model output."""
    if not text:
        return ""

    # Strategy 1: last complete fenced block
    blocks = _FENCE_BLOCK_RE.findall(text)
    if blocks:
        return blocks[-1].rstrip()

    # Strategy 2: unclosed fence (truncated output mid-block)
    open_match = _FENCE_OPEN_RE.search(text)
    if open_match:
        rest = text[open_match.end():]
        # Cut at any partial backtick run that may begin a closing fence
        rest = re.split(r"`{2,}", rest)[0]
        return rest.rstrip()

    # Strategy 3: from first Python keyword to end of text
    m = _PY_START_RE.search(text)
    if m:
        return text[m.start():].rstrip()

    # Strategy 4: pass-through (will likely fail to execute, which is correct)
    return text.rstrip()


# quick self-test
_test_inputs = [
    ("```python\ndef foo():\n    return 1\n```", "def foo():\n    return 1"),
    ("Here is reasoning...\n```\ndef a(): pass\n```\nFinal:\n```python\ndef b(): return 2\n```",
     "def b(): return 2"),
    ("Let me think.\n```python\ndef partial(x):\n    return x +",          # truncated
     "def partial(x):\n    return x +"),
    ("def naked():\n    return 42\nMore commentary",
     "def naked():\n    return 42\nMore commentary"),
    ("", ""),
]
for inp, expected in _test_inputs:
    got = extract_code(inp)
    status = "OK" if got.strip() == expected.strip() else "MISMATCH"
    print(f"  [{status}] in={inp[:40]!r}  out={got[:40]!r}")


### 5.3 Execution-based grader

We grade by running the candidate code in a **subprocess** with a hard timeout. This isolates failures (infinite loops, segfaults, recursion limits) from the notebook process.

For HumanEval, the runnable script is `prompt + completion + "\n" + test + f"\ncheck({entry_point})\n"`.

For MBPP, the runnable script is `completion + "\n" + "\n".join(test_list)`. (Each MBPP test is a stand-alone `assert` line that only requires the function to be defined.)

A subtle but important detail: HumanEval's `prompt` field already contains the function signature and docstring. If the model's extracted code *also* contains the signature (which it usually does for chat-formatted outputs), we should not concatenate `prompt + completion` — that would produce duplicate definitions. We handle both cases: detect whether the extracted code already defines `entry_point`, and only prepend `prompt` if it does not.


In [ ]:
# === subprocess-isolated grader ===

def _run_python_script(script: str, timeout: int) -> Tuple[bool, str]:
    """Execute `script` in a subprocess; return (passed, stderr_tail)."""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(script)
        script_path = f.name
    try:
        proc = subprocess.run(
            [sys.executable, script_path],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        passed = (proc.returncode == 0)
        err = (proc.stderr or proc.stdout)[-500:]
        return passed, err
    except subprocess.TimeoutExpired:
        return False, "TIMEOUT"
    except Exception as e:
        return False, f"RUNNER_ERROR: {e!r}"
    finally:
        try:
            os.unlink(script_path)
        except OSError:
            pass


def _has_def(code: str, name: str) -> bool:
    return bool(re.search(rf"^\s*(?:async\s+)?def\s+{re.escape(name)}\b", code, re.MULTILINE))


def grade_humaneval(extracted: str, problem: Dict[str, Any], timeout: int) -> Tuple[bool, str]:
    """HumanEval grader; uses problem['test'] and problem['entry_point']."""
    entry = problem["entry_point"]
    prompt = problem["prompt"]
    test = problem["test"]

    if _has_def(extracted, entry):
        # model regenerated the signature -- use only the extracted code
        body = extracted
    else:
        # model produced bare body code -- prepend prompt (which has signature)
        body = prompt + "\n" + extracted

    script = f"{body}\n\n{test}\n\ncheck({entry})\n"
    return _run_python_script(script, timeout)


def grade_mbpp(extracted: str, problem: Dict[str, Any], timeout: int) -> Tuple[bool, str]:
    """MBPP grader; uses problem['test_list']."""
    tests = problem.get("test_list", [])
    if not tests:
        return False, "NO_TESTS"
    script = extracted + "\n\n" + "\n".join(tests) + "\n"
    return _run_python_script(script, timeout)


# self-test
_p = {"prompt": "def add(a, b):\n    \"\"\"Return the sum.\"\"\"\n",
      "test": "def check(f):\n    assert f(2, 3) == 5\n    assert f(0, 0) == 0",
      "entry_point": "add"}
ok, err = grade_humaneval("    return a + b", _p, timeout=5)
print(f"  HumanEval grader self-test (bare body):     pass={ok}  err={err!r}")
ok, err = grade_humaneval("def add(a, b):\n    return a + b", _p, timeout=5)
print(f"  HumanEval grader self-test (full def):      pass={ok}  err={err!r}")

_p_mbpp = {"test_list": ["assert mul(3, 4) == 12", "assert mul(0, 5) == 0"]}
ok, err = grade_mbpp("def mul(a, b):\n    return a * b", _p_mbpp, timeout=5)
print(f"  MBPP grader self-test:                      pass={ok}  err={err!r}")


### 5.4 Prompt formatters

Two prompt formats for HumanEval:

- **`chat`** — wrap the HumanEval prompt as a user instruction with the Qwen2.5 chat template. This is the **canonical** format for an Instruct model + LoRA fine-tuned with chat-formatted training data, and we expect it to score highest.
- **`direct`** — feed the raw HumanEval prompt as a continuation (no chat template). This tests whether the LoRA fine-tuning has degraded raw completion ability. Most modern Coder benchmarks evaluate this way.

For MBPP we use only `chat` (the training distribution), with the first test case as a one-shot hint — the standard MBPP eval convention.


In [ ]:
# === prompt formatters ===

HE_INSTRUCTION = (
    "Complete the following Python function. Return ONLY the complete function "
    "definition (signature and body) inside a single ```python code block.\n\n"
)


def format_humaneval_prompt(problem: Dict[str, Any], fmt: str) -> str:
    """Build the model input for one HumanEval problem in the given format."""
    if fmt == "chat":
        user = HE_INSTRUCTION + problem["prompt"]
        messages = [{"role": "user", "content": user}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    elif fmt == "direct":
        # raw completion: pass the problem prompt as-is, model completes the body
        return problem["prompt"]
    else:
        raise ValueError(f"Unknown HumanEval prompt format: {fmt}")


def format_mbpp_prompt(problem: Dict[str, Any]) -> str:
    """Standard MBPP eval format: problem text + first test as one-shot hint."""
    text = problem["prompt"].strip() if "prompt" in problem else problem["text"].strip()
    tests = problem.get("test_list") or []
    hint = ("\nYour code should pass these tests:\n" + tests[0]) if tests else ""
    user = (
        f"{text}{hint}\n\n"
        "Return ONLY the complete function definition inside a single ```python code block."
    )
    messages = [{"role": "user", "content": user}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


### 5.5 Inference: load adapter, batch-generate, save raw outputs

The inference loop loads a base model and (optionally) a LoRA adapter, then runs greedy generation problem-by-problem. **Raw outputs are saved to JSONL before grading** — this lets us re-extract and re-grade without paying the generation cost again, and lets us audit failures directly.

For the base condition we generate without a LoRA adapter; otherwise we wrap with `PeftModel.from_pretrained`.


In [ ]:
# === inference: generate raw outputs and save them ===

def load_for_inference(adapter_dir: Optional[Path], load_in_4bit: bool = False):
    """Load base model + optional LoRA adapter, in eval mode."""
    base = load_base_model(load_in_4bit=load_in_4bit)
    if adapter_dir is not None:
        model = PeftModel.from_pretrained(base, str(adapter_dir))
    else:
        model = base
    model.eval()
    model.config.use_cache = True   # OK for eval; we're not training
    return model


@torch.inference_mode()
def generate_one(model, prompt_text: str) -> Tuple[str, bool]:
    """Greedy generate one completion. Returns (text, was_truncated)."""
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True,
                       max_length=CONFIG["max_seq_len"]).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=CONFIG["max_new_tokens"],
        do_sample=CONFIG["do_sample"],
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    truncated = (len(new_tokens) >= CONFIG["max_new_tokens"])
    return text, truncated


def run_inference(condition: str, adapter_dir: Optional[Path],
                  problems: List[Dict[str, Any]],
                  benchmark: str, fmt: str,
                  out_path: Path,
                  resume: bool = True,
                  load_in_4bit: bool = False) -> List[Dict[str, Any]]:
    """Generate completions for all problems; persist raw outputs to JSONL."""
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # If resume disabled, start fresh
    if not resume and out_path.exists():
        out_path.unlink()

    # resume: collect already-done task_ids
    done = {}
    if resume and out_path.exists():
        for l in out_path.read_text().splitlines():
            if l.strip():
                rec = json.loads(l)
                done[rec["task_id"]] = rec
        if done:
            print(f"  [resume] {len(done)}/{len(problems)} already generated; "
                  "skipping those.")

    todo = [p for p in problems if p["task_id"] not in done]
    if not todo:
        print(f"  All {len(problems)} problems already complete for {condition}/{benchmark}/{fmt}.")
        return list(done.values())

    model = load_for_inference(adapter_dir, load_in_4bit=load_in_4bit)

    new_records = []
    for p in tqdm(todo, desc=f"  gen {condition}/{benchmark}/{fmt}"):
        if benchmark == "humaneval":
            prompt_text = format_humaneval_prompt(p, fmt)
        elif benchmark == "mbpp_heldout":
            prompt_text = format_mbpp_prompt(p)
        else:
            raise ValueError(benchmark)

        text, truncated = generate_one(model, prompt_text)
        rec = {
            "task_id":       p["task_id"],
            "benchmark":     benchmark,
            "fmt":           fmt,
            "prompt_text":   prompt_text,
            "raw_output":    text,
            "truncated":     bool(truncated),
            "n_chars":       len(text),
        }
        new_records.append(rec)
        # append-as-you-go so a crash mid-loop doesn't lose progress
        with out_path.open("a") as f:
            f.write(json.dumps(rec) + "\n")

    del model
    free_gpu()

    return list(done.values()) + new_records


def grade_outputs(condition: str, raw_records: List[Dict[str, Any]],
                  problems: List[Dict[str, Any]], benchmark: str,
                  timeout: int) -> Dict[str, Any]:
    """Re-extractable grading pass over previously-generated outputs."""
    by_id = {p["task_id"]: p for p in problems}
    rows = []
    n_pass = 0
    for rec in raw_records:
        p = by_id.get(rec["task_id"])
        if p is None:
            continue
        code = extract_code(rec["raw_output"])
        if benchmark == "humaneval":
            ok, err = grade_humaneval(code, p, timeout=timeout)
        elif benchmark == "mbpp_heldout":
            ok, err = grade_mbpp(code, p, timeout=timeout)
        else:
            raise ValueError(benchmark)
        rows.append({
            "task_id":   rec["task_id"],
            "passed":    ok,
            "truncated": rec.get("truncated", False),
            "n_chars":   rec.get("n_chars", len(rec.get("raw_output", ""))),
            "err_tail":  err if not ok else "",
        })
        n_pass += int(ok)
    n = max(len(rows), 1)
    return {
        "condition":     condition,
        "benchmark":     benchmark,
        "n_total":       len(rows),
        "n_pass":        n_pass,
        "pass_at_1":     n_pass / n,
        "rows":          rows,
    }


---
## 5. Smoke test

**Run this cell first** to validate the entire pipeline (data → train → eval) on a tiny subset before committing to the full ~16-hour run. The smoke test:

- Trains one condition (`static_mix_a05`) for `smoke_max_steps=4` on `smoke_n_train=8` examples.
- Loads the resulting adapter and runs greedy generation on `smoke_n_eval=4` HumanEval problems.
- Tests the code extractor and execution-based grader.

Expected wall-clock: **~2–4 minutes** on L40S. If this works end-to-end, the full run is safe.

The smoke adapter is written to `/workspace/runs/adapters/_smoke/` and is **not** used for the real experiment.


In [ ]:
# === SMOKE TEST: run this once before committing to the full ~16-hour pipeline ===
# Trains 4 steps on 8 examples of static_mix_a05, then evaluates 4 HumanEval problems.
# Total wall-clock target: ~2-4 min on L40S.

SMOKE_DIR = CONFIG["adapters_dir"] / "_smoke"

if SMOKE_DIR.exists():
    print(f"Removing prior smoke artifacts at {SMOKE_DIR}")
    import shutil
    shutil.rmtree(SMOKE_DIR)

t0 = time.time()
smoke_summary = train_condition(
    name="static_mix_a05",
    kind="static",
    alpha=0.5,
    epochs=1,
    stages=None,
    seed=CONFIG["training_seeds"][0],
    smoke=True,
)
# point the smoke summary at a separate dir (we don't want to pollute the real one)
import shutil
real_dir = Path(smoke_summary["out_dir"])
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
for f in real_dir.iterdir():
    if f.is_file():
        shutil.copy2(f, SMOKE_DIR / f.name)
shutil.rmtree(real_dir)
print(f"\nSmoke adapter copied to {SMOKE_DIR}, training time: {time.time()-t0:.1f}s")

# now eval on 4 HumanEval problems
smoke_problems = humaneval_problems[:CONFIG["smoke_n_eval"]]
smoke_out = CONFIG["outputs_dir"] / "_smoke_he_chat.jsonl"
if smoke_out.exists(): smoke_out.unlink()

raw = run_inference(
    condition="_smoke",
    adapter_dir=SMOKE_DIR,
    problems=smoke_problems,
    benchmark="humaneval",
    fmt="chat",
    out_path=smoke_out,
    resume=False,
)
result = grade_outputs("_smoke", raw, smoke_problems, "humaneval", timeout=CONFIG["exec_timeout"])
print(f"\nSmoke eval: pass@1 = {result['pass_at_1']:.2f}  ({result['n_pass']}/{result['n_total']})")
print(f"Total smoke wall-clock: {time.time()-t0:.1f}s")

print("\n✓ If this completed without error, the full pipeline is healthy.")
print("  pass@1 itself doesn't matter for the smoke test — we only validated the plumbing.")


---
## 6. Train all conditions

This loops over the five trained-model conditions (Pure Short, Pure Long, Static α=0.2, Static α=0.5, Curriculum) for every seed in `CONFIG["training_seeds"]`. Each adapter directory is checked for an existing `adapter_model.safetensors`; matching directories are skipped.

Expected wall-clock on L40S: **~4 hours for one seed** across all five conditions.

### Adapter paths

After this cell completes, adapters live at:

```
/workspace/runs/adapters/
├── pure_short__seed42/
├── pure_long__seed42/
├── static_mix_a02__seed42/
├── static_mix_a05__seed42/
└── curriculum__seed42/
    ├── stage1/   ← intermediate (loaded by stage 2)
    ├── stage2/   ← intermediate (loaded by stage 3)
    └── stage3/   ← also mirrored at top level for convenience
```


In [ ]:
# === train all conditions x all seeds ===
training_summaries = []

for seed in CONFIG["training_seeds"]:
    for name, kind, alpha, epochs, stages in CONFIG["conditions"]:
        try:
            summary = train_condition(
                name=name, kind=kind, alpha=alpha,
                epochs=epochs, stages=stages,
                seed=seed, smoke=False,
            )
            training_summaries.append(summary)
        except Exception as e:
            print(f"\n✗ FAILED on {name} (seed={seed}): {e}")
            traceback.print_exc()
            training_summaries.append({
                "name": name, "seed": seed, "status": "failed",
                "error": str(e),
            })
            free_gpu()

# persist a roll-up
roll_path = CONFIG["results_dir"] / "training_summaries.json"
roll_path.write_text(json.dumps(training_summaries, indent=2, default=str))
print(f"\n✓ All training done. Roll-up saved to {roll_path}")
print(f"  Conditions x seeds: {len(training_summaries)}")
print(f"  Trained: {sum(1 for s in training_summaries if s.get('status') == 'trained')}")
print(f"  Skipped: {sum(1 for s in training_summaries if s.get('status') == 'skipped')}")
print(f"  Failed:  {sum(1 for s in training_summaries if s.get('status') == 'failed')}")


---
## 7. Run evaluations

For each (condition × benchmark × prompt-format), we generate raw outputs once and grade them. Raw generations are saved to `eval_outputs/{condition}__seed{S}/{benchmark}_{fmt}.jsonl` so re-grading (e.g., after improving the extractor) does not require regeneration.

The matrix:

| Condition × seed     | HumanEval (chat) | HumanEval (direct) | MBPP held-out (chat) |
|----------------------|:---:|:---:|:---:|
| base                 | ✓ | ✓ | ✓ |
| pure_short           | ✓ | ✓ | ✓ |
| pure_long            | ✓ | ✓ | ✓ |
| static_mix_a02       | ✓ | ✓ | ✓ |
| static_mix_a05       | ✓ | ✓ | ✓ |
| curriculum           | ✓ | ✓ | ✓ |

Expected wall-clock on L40S: **~12–15 hr for one seed** across all six models. Pure Long (and Curriculum-stage-3) generates the longest outputs and takes the largest share of this time.


In [ ]:
# === run all evaluations ===

# build the (condition, adapter_path) list including base
def list_eval_targets(seeds):
    targets = []
    for seed in seeds:
        targets.append(("base", seed, None))
        for name, _, _, _, _ in CONFIG["conditions"]:
            adapter_path = CONFIG["adapters_dir"] / f"{name}__seed{seed}"
            if (adapter_path / "adapter_model.safetensors").exists():
                targets.append((name, seed, adapter_path))
            else:
                print(f"⚠ skipping {name}__seed{seed} — adapter not found")
    return targets


# benchmark roster
def list_benchmarks():
    bms = []
    if CONFIG["eval_humaneval"]:
        for fmt in CONFIG["eval_prompt_formats"]:
            bms.append(("humaneval", fmt, humaneval_problems))
    if CONFIG["eval_mbpp_heldout"]:
        bms.append(("mbpp_heldout", "chat", mbpp_with_tests))
    return bms


eval_targets = list_eval_targets(CONFIG["training_seeds"])
benchmarks   = list_benchmarks()
print(f"Eval matrix: {len(eval_targets)} models × {len(benchmarks)} benchmark/format combos\n")

eval_records = []   # one row per (condition, seed, benchmark, fmt)

for cond_name, seed, adapter_path in eval_targets:
    cond_out_dir = CONFIG["outputs_dir"] / f"{cond_name}__seed{seed}"
    cond_out_dir.mkdir(parents=True, exist_ok=True)

    for benchmark, fmt, problems in benchmarks:
        out_path = cond_out_dir / f"{benchmark}_{fmt}.jsonl"
        print(f"\n=== {cond_name}/seed={seed}/{benchmark}/{fmt}  →  {out_path}")
        try:
            raw = run_inference(
                condition=cond_name,
                adapter_dir=adapter_path,
                problems=problems,
                benchmark=benchmark, fmt=fmt,
                out_path=out_path,
                resume=True,
            )
            res = grade_outputs(cond_name, raw, problems, benchmark,
                                timeout=CONFIG["exec_timeout"])
            res.update({"seed": seed, "fmt": fmt})
            eval_records.append(res)
            print(f"  pass@1 = {res['pass_at_1']:.4f}  ({res['n_pass']}/{res['n_total']})")
        except Exception as e:
            print(f"  ✗ FAILED: {e}")
            traceback.print_exc()
            eval_records.append({
                "condition": cond_name, "seed": seed,
                "benchmark": benchmark, "fmt": fmt,
                "error": str(e),
            })
            free_gpu()

# persist roll-up
roll_path = CONFIG["results_dir"] / "eval_records.json"
roll_path.write_text(json.dumps(eval_records, indent=2, default=str))
print(f"\n✓ All evaluations done. Roll-up saved to {roll_path}")


---
## 8. Secondary analyses

These analyses run on the persisted raw outputs (no regeneration needed):

1. **Output length distributions** — character counts per condition/benchmark; reveals how strongly each fine-tune adopted the long-CoT style.
2. **Truncation rates** — fraction of outputs that hit `max_new_tokens=3072`; high truncation on short-CoT conditions would indicate runaway repetition or a broken stop-token.
3. **Style adoption %** — three orthogonal signals for "did the model adopt long-CoT style":
   - presence of step markers (`Step 1:`, `First,`, `Next,`, `Then,`, `Finally,`)
   - presence of "let me think" / "let's analyze" class openers
   - output length above the short-CoT median + 1σ
4. **Pass-set Jaccard** — for each pair of conditions, the Jaccard similarity of their solved-task sets. Tells you whether different conditions solve *different* problems or the same ones (with different success rates).


In [ ]:
# === output length distributions and truncation rates ===
import matplotlib.pyplot as plt

length_records = []
for er in eval_records:
    if "rows" not in er:
        continue
    cond_dir = CONFIG["outputs_dir"] / f"{er['condition']}__seed{er['seed']}"
    raw_path = cond_dir / f"{er['benchmark']}_{er['fmt']}.jsonl"
    if not raw_path.exists():
        continue
    raws = [json.loads(l) for l in raw_path.read_text().splitlines() if l.strip()]
    for r in raws:
        length_records.append({
            "condition":  er["condition"],
            "benchmark":  er["benchmark"],
            "fmt":        er["fmt"],
            "seed":       er["seed"],
            "n_chars":    r.get("n_chars", len(r.get("raw_output", ""))),
            "truncated":  bool(r.get("truncated", False)),
            "task_id":    r["task_id"],
        })

len_df = pd.DataFrame(length_records)
print("Output-length summary (HumanEval, chat format):")
he_chat = len_df[(len_df.benchmark == "humaneval") & (len_df.fmt == "chat")]
summary = he_chat.groupby("condition").agg(
    median_chars=("n_chars", "median"),
    mean_chars=("n_chars", "mean"),
    p95_chars=("n_chars", lambda x: x.quantile(0.95)),
    truncation_rate=("truncated", "mean"),
    n=("task_id", "count"),
).round(2)
print(summary.to_string())

# plot
if not he_chat.empty:
    fig, ax = plt.subplots(1, 1, figsize=(9, 4))
    conds = sorted(he_chat.condition.unique())
    data  = [he_chat[he_chat.condition == c].n_chars.values for c in conds]
    ax.boxplot(data, labels=conds, showfliers=False)
    ax.set_ylabel("output length (chars)")
    ax.set_title("HumanEval (chat) — output length distribution by condition")
    ax.set_xticklabels(conds, rotation=30, ha="right")
    plt.tight_layout()
    fig_path = CONFIG["results_dir"] / "fig_output_lengths_he_chat.png"
    plt.savefig(fig_path, dpi=120)
    plt.show()
    print(f"\nSaved figure to {fig_path}")

len_df.to_csv(CONFIG["results_dir"] / "output_lengths.csv", index=False)


In [ ]:
# === style adoption: three orthogonal signals ===

_STEP_MARKER_RE = re.compile(
    r"\b(step\s*\d+|first[,\.]|next[,\.]|then[,\.]|finally[,\.])",
    re.IGNORECASE,
)
_THINK_OPENER_RE = re.compile(
    r"\b(let me think|let me analyze|let'?s think|let'?s analyze|i'?ll start by|"
    r"to solve this|i need to|first[,\.]|let me break)",
    re.IGNORECASE,
)


def style_signals(text: str, short_median_p84: int) -> Dict[str, int]:
    """Return three binary indicators of long-CoT style adoption."""
    return {
        "has_step_markers":  int(bool(_STEP_MARKER_RE.search(text))),
        "has_think_opener":  int(bool(_THINK_OPENER_RE.search(text[:500]))),
        "is_long":           int(len(text) > short_median_p84),
    }


# anchor: "long" means above the pure_short condition's (median + 1 sigma) on HumanEval/chat.
# This gives a per-experiment-relative threshold rather than a hardcoded one.
he_chat_short = len_df[
    (len_df.benchmark == "humaneval")
    & (len_df.fmt == "chat")
    & (len_df.condition == "pure_short")
]
if len(he_chat_short) > 0:
    short_median = float(he_chat_short.n_chars.median())
    short_std    = float(he_chat_short.n_chars.std())
    LONG_THRESHOLD = short_median + short_std
else:
    # fallback: use base condition or a default
    fallback = len_df[(len_df.benchmark == "humaneval") & (len_df.fmt == "chat")]
    LONG_THRESHOLD = float(fallback.n_chars.median()) if len(fallback) else 1000.0
print(f"Length threshold for 'is_long': {LONG_THRESHOLD:.0f} chars "
      "(pure_short median + 1 sigma on HumanEval/chat)")

style_rows = []
for er in eval_records:
    if "rows" not in er:
        continue
    cond_dir = CONFIG["outputs_dir"] / f"{er['condition']}__seed{er['seed']}"
    raw_path = cond_dir / f"{er['benchmark']}_{er['fmt']}.jsonl"
    if not raw_path.exists():
        continue
    raws = [json.loads(l) for l in raw_path.read_text().splitlines() if l.strip()]
    for r in raws:
        sig = style_signals(r.get("raw_output", ""), LONG_THRESHOLD)
        style_rows.append({
            "condition":  er["condition"],
            "benchmark":  er["benchmark"],
            "fmt":        er["fmt"],
            "seed":       er["seed"],
            "task_id":    r["task_id"],
            **sig,
        })

style_df = pd.DataFrame(style_rows)
style_summary = (
    style_df[(style_df.benchmark == "humaneval") & (style_df.fmt == "chat")]
    .groupby("condition")
    .agg(
        pct_step_markers=("has_step_markers", "mean"),
        pct_think_opener=("has_think_opener", "mean"),
        pct_long=("is_long", "mean"),
        n=("task_id", "count"),
    )
    .round(3)
)
print("\nStyle adoption on HumanEval/chat:")
print(style_summary.to_string())

style_df.to_csv(CONFIG["results_dir"] / "style_signals.csv", index=False)
style_summary.to_csv(CONFIG["results_dir"] / "style_summary.csv")


In [ ]:
# === pass-set Jaccard: do conditions solve the same or different problems? ===

def pass_set(cond, benchmark, fmt, seed):
    """Return set of task_ids passed by (cond, benchmark, fmt, seed)."""
    for er in eval_records:
        if (er.get("condition") == cond and er.get("benchmark") == benchmark
                and er.get("fmt") == fmt and er.get("seed") == seed):
            if "rows" not in er:
                return None
            return {r["task_id"] for r in er["rows"] if r.get("passed")}
    return None


def jaccard(a, b):
    if not a and not b: return 1.0
    if not a or not b:  return 0.0
    return len(a & b) / len(a | b)


# Build heatmaps: one per (benchmark, fmt) we have data for
seed = CONFIG["training_seeds"][0]
all_conds = ["base"] + [c[0] for c in CONFIG["conditions"]]

bm_combos = []
if CONFIG["eval_humaneval"]:
    for fmt in CONFIG["eval_prompt_formats"]:
        bm_combos.append(("humaneval", fmt))
if CONFIG["eval_mbpp_heldout"]:
    bm_combos.append(("mbpp_heldout", "chat"))

jaccard_tables = {}
for bm, fmt in bm_combos:
    sets = {c: pass_set(c, bm, fmt, seed) for c in all_conds}
    sets = {c: s for c, s in sets.items() if s is not None}
    conds = list(sets.keys())
    if not conds:
        continue
    M = pd.DataFrame(
        [[jaccard(sets[a], sets[b]) for b in conds] for a in conds],
        index=conds, columns=conds,
    )
    jaccard_tables[(bm, fmt)] = M
    print(f"\nPass-set Jaccard — {bm}/{fmt}:")
    print(M.round(3).to_string())

# plot one combined figure
n_panels = len(jaccard_tables)
if n_panels:
    fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 4.5), squeeze=False)
    for ax, ((bm, fmt), M) in zip(axes[0], jaccard_tables.items()):
        im = ax.imshow(M.values, vmin=0, vmax=1, cmap="viridis")
        ax.set_xticks(range(len(M))); ax.set_xticklabels(M.columns, rotation=45, ha="right")
        ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
        ax.set_title(f"{bm}/{fmt}")
        # annotate
        for i in range(len(M)):
            for j in range(len(M)):
                ax.text(j, i, f"{M.iloc[i, j]:.2f}",
                        ha="center", va="center",
                        color="white" if M.iloc[i, j] < 0.5 else "black",
                        fontsize=8)
    fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8, label="Jaccard")
    fig.suptitle("Pass-set Jaccard similarity across conditions")
    fig_path = CONFIG["results_dir"] / "fig_jaccard.png"
    plt.savefig(fig_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"\nSaved figure to {fig_path}")

# persist
for (bm, fmt), M in jaccard_tables.items():
    M.to_csv(CONFIG["results_dir"] / f"jaccard_{bm}_{fmt}.csv")


---
## 9. Final results

The cell below builds the **headline thesis table**: pass@1 by condition × benchmark × prompt-format, with delta-vs-base, and a parallel companion plot.

For interpreting the result against the diagnosis from the original setup:

- **The Small Model Learnability Gap is established** if `pure_long < pure_short` on at least one benchmark/format, with a non-trivial gap.
- **Static mixing helps** if `static_mix_a02 > pure_long` (and ideally `> pure_short`), reproducing the paper's α=0.2 finding in the code domain.
- **Curriculum helps over static mixing** if `curriculum > max(static_mix_*)` — the central thesis claim.
- **Curriculum's specific schedule matters** comparing `curriculum` (0→50→80) to `static_mix_a05` (mid-static) and `static_mix_a02` (paper-optimum static) — the differences localize whether the gain comes from the *trajectory* or just the *endpoint*.

For multi-seed runs, this cell additionally reports mean ± std across seeds.


In [ ]:
# === final results table + figure ===

# Long-format DataFrame of pass@1 across (condition, seed, benchmark, fmt)
rows = []
for er in eval_records:
    if "pass_at_1" not in er:
        continue
    rows.append({
        "condition":  er["condition"],
        "seed":       er["seed"],
        "benchmark":  er["benchmark"],
        "fmt":        er["fmt"],
        "pass_at_1":  er["pass_at_1"],
        "n_pass":     er["n_pass"],
        "n_total":    er["n_total"],
    })
results_df = pd.DataFrame(rows)
results_df.to_csv(CONFIG["results_dir"] / "results_long.csv", index=False)

# Pivoted wide table — one row per condition, one column per (benchmark, fmt)
def _label(bm, fmt):
    if bm == "humaneval":  return f"HE/{fmt}"
    if bm == "mbpp_heldout": return "MBPP-held"
    return f"{bm}/{fmt}"

agg = (
    results_df.groupby(["condition", "benchmark", "fmt"])
    .agg(mean=("pass_at_1", "mean"), std=("pass_at_1", "std"), n=("seed", "nunique"))
    .reset_index()
)
agg["col"] = agg.apply(lambda r: _label(r.benchmark, r.fmt), axis=1)
agg["cell"] = agg.apply(
    lambda r: (f"{r['mean']*100:.2f}" + (f" ± {r['std']*100:.2f}" if r["n"] > 1 else "")),
    axis=1,
)

ordered_conds = ["base"] + [c[0] for c in CONFIG["conditions"]]
ordered_cols  = []
if CONFIG["eval_humaneval"]:
    for fmt in CONFIG["eval_prompt_formats"]:
        ordered_cols.append(_label("humaneval", fmt))
if CONFIG["eval_mbpp_heldout"]:
    ordered_cols.append(_label("mbpp_heldout", "chat"))

wide = agg.pivot(index="condition", columns="col", values="cell").reindex(
    index=ordered_conds, columns=ordered_cols
)
print("\n=== HEADLINE RESULTS — pass@1 (%) ===\n")
print(wide.fillna("—").to_string())

# delta-vs-base table (numeric, percentage points)
mean_only = agg.pivot(index="condition", columns="col", values="mean").reindex(
    index=ordered_conds, columns=ordered_cols
)
if "base" in mean_only.index:
    delta = (mean_only - mean_only.loc["base"]) * 100
    print("\n=== DELTA vs BASE (percentage points) ===\n")
    print(delta.round(2).fillna("—").to_string())
    delta.to_csv(CONFIG["results_dir"] / "delta_vs_base.csv")

wide.to_csv(CONFIG["results_dir"] / "results_wide.csv")
mean_only.to_csv(CONFIG["results_dir"] / "results_means.csv")

# Headline figure
n_groups = len(ordered_cols)
n_conds  = len(ordered_conds)
fig, ax = plt.subplots(1, 1, figsize=(max(8, 1.6 * n_groups * n_conds / 4), 5))
x = np.arange(n_groups)
width = 0.8 / n_conds
for i, cond in enumerate(ordered_conds):
    if cond not in mean_only.index:
        continue
    vals = [mean_only.loc[cond, c] * 100 if not pd.isna(mean_only.loc[cond, c]) else 0
            for c in ordered_cols]
    ax.bar(x + i * width - 0.4 + width / 2, vals, width, label=cond)
ax.set_xticks(x); ax.set_xticklabels(ordered_cols)
ax.set_ylabel("pass@1 (%)")
ax.set_title("pass@1 by condition × benchmark/format")
ax.legend(loc="best", fontsize=8, ncol=2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig_path = CONFIG["results_dir"] / "fig_headline.png"
plt.savefig(fig_path, dpi=120)
plt.show()
print(f"\n✓ Figure saved to {fig_path}")
print(f"✓ All result CSVs saved to {CONFIG['results_dir']}")


---
## 10. Where to go from here

### Interpreting the results in your thesis

Read the four signals together — no single one is decisive on its own:

1. **Headline pass@1** (Section 9) — the primary outcome measure.
2. **Pass-set Jaccard** (Section 8) — if the curriculum's gain comes from solving *different* problems than static mixing, that is a substantively different finding from solving the same problems more reliably.
3. **Style adoption** (Section 8) — confirms that long-CoT training conditions actually changed model behavior. If `pure_long` and `static_mix_a05` show similar style adoption %, the difference in pass@1 is a clean signal of *learnability*, not *exposure*.
4. **Output length distributions** (Section 8) — the truncation rate is a sanity check; if any condition's truncation rate is high, its pass@1 is a *lower bound*.

### If you have time for variance estimates

Add seeds to `CONFIG["training_seeds"]` (e.g. `[42, 1337]`), then re-run Sections 6 and 7. Resume support means already-trained adapters and already-generated outputs are not redone. A second seed for just the two key conditions (`static_mix_a02` and `curriculum`) gives you a noise estimate on the contrast that matters most without doubling cost.

### If results are inconclusive

If pass@1 differences across conditions are within ±1 pp on HumanEval, consider:

- **Larger evaluation set**: enable HumanEval+ via `CONFIG["eval_humaneval_plus"] = True` to triple the test surface.
- **Longer training**: bump `epochs` to 5 (or curriculum to 2/2/1 stages) — you may be under-fitting.
- **Different scale**: try `Qwen/Qwen2.5-0.5B-Instruct` for an even-smaller-model condition. The Learnability Gap should be *more* pronounced at smaller scale, which is a clean way to demonstrate the phenomenon.

### Files this notebook produces

```
/workspace/runs/
├── adapters/                 # one LoRA adapter directory per (condition, seed)
├── eval_outputs/             # raw generations, JSONL, append-on-write so re-grading is free
└── results/
    ├── training_summaries.json
    ├── eval_records.json
    ├── results_long.csv      # one row per (cond, seed, benchmark, fmt)
    ├── results_wide.csv      # publication-style table with mean ± std
    ├── results_means.csv
    ├── delta_vs_base.csv
    ├── output_lengths.csv
    ├── style_signals.csv
    ├── style_summary.csv
    ├── jaccard_*.csv
    ├── fig_output_lengths_he_chat.png
    ├── fig_jaccard.png
    └── fig_headline.png
```

All CSVs are in tidy long-or-pivoted form so you can drop them into your thesis with `pandas.read_csv` + `to_latex`.
